# Quick personalized K-channel seizure forecast

This is the one-patient smoke-test companion to `personalized_channels.ipynb`. It uses the same chronological split, compact channel-local features, fixed-K selection, calibrated rolling discrete-hazard model, a simple binary-risk comparator, and untouched test evaluation. It disables swap refinement and runs only one random-channel baseline. Cross-channel referencing is omitted so selected features cannot contain information from excluded electrodes.

> **Research only:** do not use these forecasts for clinical decisions.

## 1. Quick-run settings

Change `K` before running. Leave `PATIENT_ID = None` to use the first eligible patient, or enter a Siena identifier such as `'PN00'`.

In [1]:
K = 4
PATIENT_ID = "PN05"
FORCE_REBUILD_FEATURES = False

In [2]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / 'personalized_channels_workflow.py').exists():
    candidates = list(NOTEBOOK_DIR.rglob('personalized_channels_workflow.py'))
    if len(candidates) != 1:
        raise FileNotFoundError('Run from the scripts directory or repository root.')
    NOTEBOOK_DIR = candidates[0].parent
sys.path.insert(0, str(NOTEBOOK_DIR))

import rolling_seizure_forecasting as rsf
import personalized_channels_workflow as pc

paths = pc.personalized_paths(NOTEBOOK_DIR)
forecast_config = rsf.ForecastConfig(test_fraction=0.20, max_iter=60)
config = pc.PersonalizedConfig(
    k=K, patient_ids=None if PATIENT_ID is None else (PATIENT_ID,),
    random_baseline_repeats=1, swap_refinement=False, max_patients=1,
    force_rebuild_features=FORCE_REBUILD_FEATURES, max_threshold_folds=2,
    quick_mode=True,
)
config.validate()
print(f'Quick mode: K={K}; requested patient={PATIENT_ID or "first eligible"}')
print(f'Results will be saved to {paths["quick_results"]}')

Quick mode: K=4; requested patient=PN05
Results will be saved to C:\Users\sahil\Documents\cosmos\26-the-optimizers-analysis\final_project\results\personalized_channels_quick


## 2. Pick one eligible patient

Eligibility requires at least two usable seizures and at least K consistently named EEG channels. With two seizures, the first seizure/control pair trains the model and the second pair tests it. The training-only channel ranking is then necessarily labeled as a resubstitution fallback.

In [3]:
manifest = pc.load_manifest(paths, forecast_config)
counts = (
    manifest.loc[manifest['episode_type'].eq('preictal')]
    .groupby('patient_id')['source_event_id'].nunique()
    .sort_index()
)
eligible = counts.loc[counts.ge(2)]
if PATIENT_ID is None:
    if eligible.empty:
        raise ValueError('No patient has at least two usable seizures.')
    selected_patient = str(eligible.index[0])
    config = pc.PersonalizedConfig(
        k=K, patient_ids=(selected_patient,), random_baseline_repeats=1,
        swap_refinement=False, max_patients=1,
        force_rebuild_features=FORCE_REBUILD_FEATURES, max_threshold_folds=2,
        quick_mode=True,
    )
else:
    selected_patient = PATIENT_ID
if selected_patient not in eligible.index:
    raise ValueError(f'{selected_patient} does not have at least two usable seizures.')
display(counts.rename('usable_seizures').to_frame())
print(f'Running quick analysis for {selected_patient}')

,usable_seizures
patient_id,
PN00,5
PN01,2
PN03,2
PN05,3
PN06,5
PN07,1
PN09,3
PN10,10
PN11,1


Running quick analysis for PN05


## 3. Run and inspect

The first feature-extraction run may still take several minutes because the EEG must be read and transformed. The cache makes later quick and full runs much faster.

In [4]:
summary, selection_trace, test_predictions = pc.run_analysis(
    manifest, paths, config, forecast_config
)
display(summary.T)
display(selection_trace)
display(test_predictions.head(12))

risk_curve = test_predictions.groupby(
    ['model', 'episode_type', 'landmark_step'], as_index=False
)['event_risk_5m'].mean()
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True, sharey=True)
for axis, (model_name, group) in zip(axes.flat, risk_curve.groupby('model')):
    for episode_type, line in group.groupby('episode_type'):
        axis.plot(line['landmark_step'] * 5 / 60, line['event_risk_5m'], label=episode_type)
    axis.set_title(model_name.replace('_', ' '))
    axis.set(xlabel='Minutes into held-out episode', ylabel='Mean predicted 5-minute risk', ylim=(0, 1))
    axis.legend()
fig.tight_layout()
plt.show()

PN05 [1/6] PN05_interictal_01
PN05 [2/6] PN05_interictal_02
PN05 [3/6] PN05_interictal_03
PN05 [4/6] PN05_S01_preictal
PN05 [5/6] PN05_S02_preictal
PN05 [6/6] PN05_S03_preictal


,0
patient_id,PN05
status,included
requested_k,4
available_channels,31
selected_channels,"FP1, PZ, FP2, P3"
selection_validation,expanding_chronological_validation
n_seizures,3
n_train_seizures,2
n_test_seizures,1
train_event_ids,"[PN05_S01, PN05_S02]"


,patient_id,step,action,channel,channels,selection_score,validation_auprc,validation_brier
0,PN05,1,add,FP1,FP1,0.844986,0.884310,0.157298
1,PN05,2,add,PZ,"FP1, PZ",0.874125,0.967999,0.375495
2,PN05,3,add,FP2,"FP1, PZ, FP2",0.917785,0.967854,0.200275
3,PN05,4,add,P3,"FP1, PZ, FP2, P3",0.749359,0.806696,0.229348


,patient_id,source_event_id,episode_id,episode_type,recording,landmark_step,time_to_event_seconds,has_event_in_5m,event_risk_5m,warning,model
0,PN05,PN05_S03,PN05_interictal_03,interictal,PN05-4.edf,0,NaN,0,0.228961,False,selected_k
1,PN05,PN05_S03,PN05_interictal_03,interictal,PN05-4.edf,1,NaN,0,0.194613,False,selected_k
2,PN05,PN05_S03,PN05_interictal_03,interictal,PN05-4.edf,2,NaN,0,0.173580,False,selected_k
3,PN05,PN05_S03,PN05_interictal_03,interictal,PN05-4.edf,3,NaN,0,0.354246,False,selected_k
4,PN05,PN05_S03,PN05_interictal_03,interictal,PN05-4.edf,4,NaN,0,0.410265,False,selected_k
5,PN05,PN05_S03,PN05_interictal_03,interictal,PN05-4.edf,5,NaN,0,0.632878,False,selected_k
6,PN05,PN05_S03,PN05_interictal_03,interictal,PN05-4.edf,6,NaN,0,0.998823,False,selected_k
7,PN05,PN05_S03,PN05_interictal_03,interictal,PN05-4.edf,7,NaN,0,0.986851,False,selected_k
8,PN05,PN05_S03,PN05_interictal_03,interictal,PN05-4.edf,8,NaN,0,0.999107,False,selected_k
9,PN05,PN05_S03,PN05_interictal_03,interictal,PN05-4.edf,9,NaN,0,0.995744,False,selected_k


## 4. What to check before the full run

- `status` should be `included`.
- `selected_channels` should contain exactly K names.
- `n_train_seizures` and `n_test_seizures` should match the chronological 80/20 rule.
- `selection_validation` documents whether true chronological validation was possible.
- Compare `selected_auprc` with `test_no_skill_auprc`, `all_auprc`, and `random_k_mean_auprc`, then compare those hazard results with `binary_selected_auprc`.
- Confirm `selected_threshold_method` is `chronological_oof` and inspect the four risk-over-time panels for separation rather than saturated probabilities.

Artifacts are saved under `results/personalized_channels_quick/`. Running this notebook does not modify raw EDF files.